## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
)


# 에이전트 메모리(Agent Memory)

이 노트북은 agent가 왜 메모리를 필요로 하는지, 그리고 메모리를 아무거나 무한정 저장하면 왜 오히려 품질이 나빠질 수 있는지 단계적으로 보여준다. 단기 메모리(short-term)는 방금 대화한 내용, 장기 메모리(long-term)는 JSON에 남겨 둘 사용자 선호나 사실, 벡터 메모리(vector memory)는 의미적으로 비슷한 기억을 꺼내기 위한 검색 레이어라고 생각하면 이해가 쉽다.

## 학습 목표
- short-term, long-term, vector memory의 차이를 일상 비유와 코드 구조 둘 다로 설명할 수 있다.
- memory retrieval이 document retrieval과 어떻게 결합되는지 이해한다.
- selective memory update가 왜 필요한지, "모든 대화를 다 기억하면 안 되는 이유"를 말할 수 있다.
- memory-aware workflow가 trace에 어떤 흔적을 남기는지 읽을 수 있다.


## 개념 설명

사람도 모든 정보를 한 번에 같은 방식으로 기억하지 않는다. 방금 들은 말은 작업 기억처럼 잠깐 들고 있고, 자주 반복되는 취향이나 규칙은 오래 저장하며, 특정 질문을 받으면 과거 경험 중 관련된 것만 떠올린다. agent 메모리도 이 세 가지를 각각 short-term, long-term, vector memory로 분리해 생각하면 훨씬 직관적이다.

중요한 점은 메모리가 많다고 무조건 좋은 것이 아니라는 사실이다. 관련 없는 기억이 많이 섞이면 retrieval noise가 늘고, 오래된 규칙이 최신 사실을 덮을 수도 있다. 그래서 update policy와 retrieval policy를 함께 설계해야 한다.

**목적**
- 메모리를 저장 장치가 아니라 reasoning 보조 장치로 이해한다.

**핵심 로직**
- short-term은 최근 대화 버퍼다.
- long-term은 지속 저장되는 구조화 기억이다.
- vector memory는 의미 기반 검색을 위한 기억 인덱스다.

**결과 해석 가이드**
- 이 notebook의 핵심 질문은 "무엇을 기억할 것인가"보다 "어떻게 기억을 통제할 것인가"다.

**💡 면접 포인트**
- "Memory는 context window를 무한 확장하는 장치가 아니라, 필요한 정보를 선별적으로 다시 주입하는 장치"라고 설명할 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

아래 셀은 memory 실험에 필요한 클래스와 helper를 한 번에 불러온다. 여기서 주목할 부분은, 메모리 구현이 notebook 안에 숨어 있지 않고 `src/memory.py`와 `src/workflow.py`에 모여 있다는 점이다. 그래야 memory behavior를 평가, trace, workflow와 일관되게 연결할 수 있다.

**목적**
- 메모리 저장소와 workflow helper를 초기화한다.

**핵심 로직**
- 장기 메모리 저장 파일이 남아 있으면 삭제해 실험을 재현 가능하게 만든다.
- retriever와 workflow helper를 함께 불러와 나중에 문서+기억 통합 실험을 할 수 있게 한다.

**결과 해석 가이드**
- 이 셀은 환경 초기화 단계라 출력보다 재현성 확보가 목적이다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.config import get_paths
from src.ingestion import build_demo_index
from src.memory import (
    LongTermMemory,
    ShortTermMemory,
    VectorMemory,
    build_memory_augmented_context,
    display_memory,
    score_memory_importance,
    selective_memory_update,
    summarize_memory_events,
)
from src.utils import display_trace
from src.workflow import run_workflow_with_memory

pd.set_option('display.max_colwidth', 140)
paths = get_paths()
memory_store_path = paths.logs_dir / 'agent_memory_notebook_store.json'
if memory_store_path.exists():
    memory_store_path.unlink()


## 소개(Introduction)

본격적인 메모리 실험에 들어가기 전에, 이 notebook에서 short-term memory를 "대화 버퍼", long-term memory를 "저장된 프로필/사실", vector memory를 "의미 유사 기억 검색기"로 읽겠다고 먼저 고정해 두자. 이렇게 비유를 잡아두면 뒤에서 각 클래스 메서드가 왜 그렇게 생겼는지 이해하기 쉬워진다.

**결과 해석 가이드**
- 단기 메모리는 최근 맥락 유지, 장기 메모리는 지속성, 벡터 메모리는 재호출(retrieval)에 강하다.


## 단기 메모리(Short-Term Memory)

아래 셀은 최근 대화 기록을 버퍼처럼 쌓는 short-term memory를 실험한다. short-term memory의 목적은 agent가 방금 주고받은 문맥을 잃지 않게 하는 것이다. 예를 들어 "앞으로 답변은 짧게" 같은 지시는 문서에 없지만, 직전 대화에서는 매우 중요할 수 있다.

**목적**
- 최근 대화 이벤트를 순서대로 저장하고, 버퍼 길이를 제한하는 구조를 이해한다.

**핵심 로직**
- `append_event()`가 event를 쌓고 timestamp를 붙인다.
- `max_items`가 있으면 오래된 이벤트부터 자동으로 떨어져 나간다.
- `to_frame()`은 버퍼 상태를 표로 보여줘 inspectability를 높인다.

**주요 파라미터/변수**
- `max_items=6`: 버퍼 최대 크기
- `kind`: user/assistant 같은 이벤트 종류
- `metadata`: 부가 맥락

**실제 소스 코드: ShortTermMemory — src/memory.py**
```python
class ShortTermMemory:
    def __init__(self, max_items: int | None = None) -> None:
        self.max_items = max_items
        self.events: list[dict[str, Any]] = []

    def append_event(
        self,
        kind: str,
        content: str,
        metadata: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        event = {
            "event_id": len(self.events) + 1,
            "kind": kind,
            "content": normalize_text(content),
            "metadata": metadata or {},
            "timestamp": iso_timestamp(),
        }
        self.events.append(event)
        if self.max_items is not None and len(self.events) > self.max_items:
            self.events = self.events[-self.max_items :]
        return event

    def last_n(self, limit: int) -> list[dict[str, Any]]:
        return self.events[-limit:]

    def clear(self) -> None:
        self.events.clear()

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.events, columns=["event_id", "kind", "content", "metadata", "timestamp"])
```

**코드 읽기 포인트**
- `event_id`는 append 순서대로 증가해 대화 흐름을 다시 읽기 쉽게 만든다.
- `content`를 `normalize_text()`로 정규화해 비교와 검색 품질을 높인다.
- `self.events = self.events[-self.max_items:]`는 버퍼 초과 시 최근 이벤트만 남기는 핵심 줄이다.
- 즉 short-term memory는 영구 저장소가 아니라 "최근 작업 컨텍스트"다.

**결과 해석 가이드**
- 표에서 event 순서가 유지되면 버퍼가 정상이다.
- `max_items`보다 많은 event를 넣었을 때 오래된 항목이 자연스럽게 사라지는지도 중요하다.


In [ ]:
short_term = ShortTermMemory(max_items=6)
short_term.append_event('user', 'Can you explain the rollout timeline?')
short_term.append_event('assistant', 'The pilot runs from March 10, 2025 to April 4, 2025.')
short_term.append_event('user', 'Please keep future answers concise.')
short_term.append_event('assistant', 'Noted. I will keep answers concise when possible.')
short_term.to_frame()


## 최근 메모리 다시 읽기

memory를 저장하는 것만큼 중요한 것이 retrieval이다. short-term memory는 보통 전체 버퍼를 다 쓰기보다 최근 몇 턴만 잘라 쓰는 경우가 많다. agent는 늘 모든 대화를 볼 필요가 없고, 오히려 너무 많이 보면 attention을 낭비한다.

**목적**
- 최근 N개 이벤트만 다시 읽는 패턴을 본다.

**핵심 로직**
- `last_n(3)`은 최근 3개 이벤트만 가져온다.

**주요 파라미터/변수**
- `limit`: 다시 읽을 최근 이벤트 수

**결과 해석 가이드**
- 반환된 표가 최신 대화 위주로 잘리는지 보면 short-term retrieval이 의도대로 동작하는지 알 수 있다.


In [ ]:
recent_turns = short_term.last_n(3)
pd.DataFrame(recent_turns)


## 장기 메모리(Long-Term Memory)

장기 메모리는 "사용자 취향"이나 "반복해서 써야 할 사실"처럼 세션이 끝나도 남아야 하는 정보를 담당한다. 이 저장소는 이를 가장 단순한 JSON persistence로 구현한다. 교육용 설계에서 이 선택이 좋은 이유는, 저장 형식이 투명해서 memory corruption이나 overwrite 문제를 바로 확인할 수 있기 때문이다.

**목적**
- 메모리를 디스크에 저장하고 다시 불러오는 persistent layer를 이해한다.

**핵심 로직**
- 초기화 시 `_load()`로 기존 JSON을 읽는다.
- `store_memory()`는 key 단위 upsert를 수행한다.
- `_persist()`가 매번 파일에 저장해 세션 종료 후에도 남긴다.

**주요 파라미터/변수**
- `key`: 장기 메모리 식별자
- `category`: preference, fact 등 메모리 종류
- `importance`: 나중에 중요도 기반 필터링을 붙일 여지를 남긴다.

**실제 소스 코드: LongTermMemory — src/memory.py**
```python
class LongTermMemory:
    def __init__(self, path: Path) -> None:
        self.path = Path(path)
        self._items: list[dict[str, Any]] = self._load()

    def _load(self) -> list[dict[str, Any]]:
        if not self.path.exists():
            return []
        payload = read_json(self.path)
        return list(payload) if isinstance(payload, list) else []

    def _persist(self) -> None:
        ensure_directory(self.path.parent)
        write_json(self.path, self._items)

    def store_memory(
        self,
        key: str,
        value: str,
        category: str = "fact",
        metadata: dict[str, Any] | None = None,
        importance: float | None = None,
    ) -> dict[str, Any]:
        item = {
            "key": key,
            "value": normalize_text(value),
            "category": category,
            "metadata": metadata or {},
            "importance": importance,
            "updated_at": iso_timestamp(),
        }
        existing_index = next((index for index, entry in enumerate(self._items) if entry["key"] == key), None)
        if existing_index is None:
            self._items.append(item)
        else:
            self._items[existing_index] = item
        self._persist()
        return item

    def retrieve_memory(self, key: str) -> dict[str, Any] | None:
        return next((item for item in self._items if item["key"] == key), None)

    def list_items(self) -> list[dict[str, Any]]:
        return list(self._items)

    def search(self, query: str, limit: int = 3) -> list[dict[str, Any]]:
        query_tokens = content_tokens(query)
        matches: list[dict[str, Any]] = []
        for item in self._items:
            searchable_text = f"{item['key']} {item['value']} {item['category']}"
            lexical_score = overlap_ratio(query_tokens, content_tokens(searchable_text))
            exact_key_bonus = 0.25 if item["key"].lower() in normalize_text(query).lower() else 0.0
            score = round(min(1.0, lexical_score + exact_key_bonus), 3)
            if score <= 0.0:
                continue
            matches.append(
                {
                    "memory_type": "long_term",
                    "key": item["key"],
                    "text": item["value"],
                    "score": score,
                    "category": item["category"],
                    "metadata": item["metadata"],
                    "timestamp": item["updated_at"],
                }
            )
        return sorted(matches, key=lambda entry: entry["score"], reverse=True)[:limit]

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self._items, columns=["key", "value", "category", "importance", "updated_at"])
```

**코드 읽기 포인트**
- `existing_index`를 찾는 부분이 overwrite/upsert 핵심이다.
- `updated_at`을 남겨 최신성 판단 근거를 만든다.
- `list_items()`와 `to_frame()`이 있어서 저장 내용을 사람이 직접 검증할 수 있다.
- 즉 long-term memory는 "기억"이면서 동시에 "감사 가능한 저장소"다.

**결과 해석 가이드**
- 재로딩 후에도 같은 key/value가 보이면 persistence가 정상이다.
- category와 importance가 함께 저장되면 이후 selective retrieval/update 정책을 붙이기 쉬워진다.


In [ ]:
long_term = LongTermMemory(memory_store_path)
long_term.store_memory('mina_answer_style', 'Mina prefers concise answers that lead with exact dates.', category='preference')
long_term.store_memory('apollo_pilot_city', 'Team Apollo is piloting the scheduling template in Seoul.', category='fact')
reloaded_long_term = LongTermMemory(memory_store_path)
reloaded_long_term.to_frame()


## 저장한 장기 메모리 조회하기

저장이 됐다고 retrieval이 쉬워지는 것은 아니다. 장기 메모리는 보통 key 기반 조회와 검색 기반 조회를 모두 갖는데, 이 셀은 그중 가장 단순한 key lookup을 보여준다. 정확히 어떤 선호나 규칙을 꺼내야 할 때 유용하다.

**목적**
- 장기 메모리에서 특정 key를 정확히 찾는다.

**핵심 로직**
- `retrieve_memory(key)`는 내부 리스트에서 key가 일치하는 첫 항목을 반환한다.

**결과 해석 가이드**
- 반환값이 dict 하나라는 점이 중요하다. 벡터 검색처럼 후보 여러 개가 아니라, 확정된 프로필 값을 읽을 때 쓰기 좋다.


In [ ]:
reloaded_long_term.retrieve_memory('mina_answer_style')


## 벡터 메모리(Vector Memory)

vector memory는 "정확한 key는 모르지만 의미상 비슷한 기억을 찾고 싶다"는 문제를 해결한다. 예를 들어 사용자가 "Mina에게 rollout timing을 어떻게 설명하면 좋지?"라고 물었을 때, 과거에 저장한 "Mina prefers concise rollout answers..." 같은 기억을 semantic retrieval로 끌어올 수 있다.

**목적**
- 텍스트 메모리를 TF-IDF 기반 의미 유사 검색 대상으로 만든다.

**핵심 로직**
- `add_memory()`가 entry를 쌓고, 추가될 때마다 `_rebuild_index()`로 vectorizer와 matrix를 갱신한다.
- `search()`는 cosine similarity 85%, lexical overlap 15%를 섞어 점수를 계산한다.
- importance는 검색 점수와 별개로 기억의 상대적 중요도를 보조 신호로 남긴다.

**주요 파라미터/변수**
- `importance`: 메모리 자체의 중요도
- `top_k`: 가져올 상위 기억 수
- `min_score`: 너무 약한 기억을 걸러내는 기준

**실제 소스 코드: VectorMemory — src/memory.py**
```python
class VectorMemory:
    def __init__(self) -> None:
        self.entries: list[dict[str, Any]] = []
        self.vectorizer: TfidfVectorizer | None = None
        self.matrix: Any = None

    def _rebuild_index(self) -> None:
        if not self.entries:
            self.vectorizer = None
            self.matrix = None
            return
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        self.matrix = self.vectorizer.fit_transform(entry["text"] for entry in self.entries)

    def add_memory(
        self,
        text: str,
        metadata: dict[str, Any] | None = None,
        importance: float = 0.5,
    ) -> dict[str, Any]:
        entry = {
            "memory_id": f"memory_{len(self.entries) + 1}",
            "text": normalize_text(text),
            "metadata": metadata or {},
            "importance": round(float(importance), 3),
            "timestamp": iso_timestamp(),
        }
        self.entries.append(entry)
        self._rebuild_index()
        return entry

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0) -> list[dict[str, Any]]:
        if not self.entries or self.vectorizer is None or self.matrix is None:
            return []

        query_vector = self.vectorizer.transform([query])
        cosine_scores = cosine_similarity(query_vector, self.matrix).flatten()
        lexical_scores = [
            overlap_ratio(content_tokens(query), content_tokens(entry["text"]))
            for entry in self.entries
        ]

        ranked: list[dict[str, Any]] = []
        for index, entry in enumerate(self.entries):
            score = round(float((cosine_scores[index] * 0.85) + (lexical_scores[index] * 0.15)), 4)
            if score < min_score:
                continue
            ranked.append(
                {
                    "memory_type": "vector",
                    "memory_id": entry["memory_id"],
                    "text": entry["text"],
                    "score": score,
                    "importance": entry["importance"],
                    "metadata": entry["metadata"],
                    "timestamp": entry["timestamp"],
                }
            )

        return sorted(ranked, key=lambda entry: entry["score"], reverse=True)[:top_k]

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.entries, columns=["memory_id", "text", "importance", "metadata", "timestamp"])
```

**코드 읽기 포인트**
- `TfidfVectorizer(ngram_range=(1, 2), stop_words="english")`: 현재 구현은 영문 친화적이라 한국어에서는 한계가 있을 수 있다.
- `_rebuild_index()`가 add 시마다 전체 인덱스를 다시 만든다. 작은 데모엔 충분하지만 대규모 시스템에서는 비용이 커질 수 있다.
- `score = cosine*0.85 + lexical*0.15`: semantic retrieval과 표면 단어 일치를 함께 본다.
- 벡터 메모리는 "정확한 slot lookup"이 아니라 "관련 기억 후보 추천"에 가깝다.

**결과 해석 가이드**
- 상위 검색 결과가 질문과 의미상 맞는지 보면 vector memory 품질을 직관적으로 읽을 수 있다.
- importance가 높아도 relevance가 낮으면 상위에 못 올라올 수 있다는 점도 중요하다.

**💡 면접 포인트**
- "Vector memory는 long-term memory를 semantic access 가능하게 만든 레이어"라고 말하면 이해가 쉽다.


In [ ]:
vector_memory = VectorMemory()
vector_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
vector_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
vector_memory.add_memory('The pilot retrospective is scheduled for June.', metadata={'kind': 'event'}, importance=0.5)
vector_search_results = vector_memory.search('How should I answer rollout timing for Mina?', top_k=3)
pd.DataFrame(vector_search_results)


## Agent에서의 메모리 검색(Memory Retrieval in Agents)

agent가 실제로 기억을 쓰려면, 문서 retrieval과 memory retrieval을 어떻게 합칠지 결정해야 한다. 이 저장소는 문서와 기억을 함께 묶은 `combined_context`를 만들어, 필요하면 메모리를 문서처럼 취급해 downstream node에 넣을 수 있게 설계했다.

**목적**
- 질문 시점에 관련 기억만 선별적으로 꺼내는 흐름을 이해한다.

**핵심 로직**
- `retrieve_relevant_memories()`는 short-term, long-term, vector memory를 모두 훑어 관련 후보를 만든다.
- 후보를 score 기준으로 정렬해 상위 k개만 남긴다.
- `build_memory_augmented_context()`는 문서와 메모리를 하나의 context bundle로 묶는다.

**주요 파라미터/변수**
- `top_k=3`: 너무 많은 기억을 넣지 않기 위한 절제 장치
- `retrieved_docs`: 기본 문서 근거
- `retrieved_memories`: 메모리 보강 근거

**실제 소스 코드: retrieve_relevant_memories() — src/memory.py**
```python
def retrieve_relevant_memories(
    query: str,
    short_term_memory: ShortTermMemory | None = None,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    top_k: int = 3,
) -> list[dict[str, Any]]:
    matches: list[dict[str, Any]] = []
    query_tokens = content_tokens(query)

    if short_term_memory is not None:
        for event in short_term_memory.last_n(top_k * 2):
            score = round(min(1.0, overlap_ratio(query_tokens, content_tokens(event["content"])) + 0.2), 3)
            if score <= 0.0:
                continue
            matches.append(
                {
                    "memory_type": "short_term",
                    "key": f"event_{event['event_id']}",
                    "text": event["content"],
                    "score": score,
                    "category": event["kind"],
                    "metadata": event["metadata"],
                    "timestamp": event["timestamp"],
                }
            )

    if long_term_memory is not None:
        matches.extend(long_term_memory.search(query, limit=top_k))

    if vector_memory is not None:
        matches.extend(vector_memory.search(query, top_k=top_k))

    ranked = sorted(matches, key=lambda entry: (entry["score"], entry.get("importance", 0.0)), reverse=True)
    return ranked[:top_k]
```

**코드 읽기 포인트**
- short-term memory는 최근 event에서 overlap 비율에 0.2 보너스를 더해 즉시성(recency)을 반영한다.
- long-term search와 vector search 결과를 한 리스트로 합친 뒤 다시 랭킹한다.
- 즉 메모리 retrieval은 단순 append가 아니라, 여러 memory channel의 late fusion이라고 볼 수 있다.

**결과 해석 가이드**
- `memory_count`가 0이면 문서만으로 답하게 된다.
- `memory_texts`가 질문 의도와 맞으면 memory augmentation이 제대로 작동하는 것이다.


In [ ]:
retriever = build_demo_index(persist=False)
retrieved_docs = retriever.search('When does the organization-wide rollout begin?', top_k=3)
context_bundle = build_memory_augmented_context(
    query='How should I answer rollout timing for Mina?',
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
{
    'doc_sources': [doc['source'] for doc in context_bundle['retrieved_docs']],
    'memory_count': len(context_bundle['retrieved_memories']),
    'memory_texts': [memory['text'] for memory in context_bundle['retrieved_memories']],
}


## 메모리 업데이트 전략(Memory Update Strategy)

모든 대화를 다 기억하면 안 되는 이유를 가장 직접적으로 보여주는 셀이 바로 이 부분이다. 중요하지 않은 잡담, 일회성 감탄, 이미 obsolete된 정보까지 다 저장하면 vector memory가 오염되고, retrieval noise가 늘어나며, 잘못된 오래된 규칙이 새 답변에 끼어들 수 있다.

**목적**
- 무엇을 남기고 무엇을 버릴지 결정하는 importance scoring을 이해한다.

**핵심 로직**
- `score_memory_importance()`는 길이, 키워드, 숫자 포함 여부, category, source를 보고 점수를 준다.
- 점수는 threshold와 비교돼 저장 여부를 결정하는 입력이 된다.

**주요 파라미터/변수**
- `threshold`: 저장 허용선
- `category`: preference/fact/constraint 같은 의미 구분
- `metadata.source`: user 발화인지 agent 생성인지 구분

**실제 소스 코드: score_memory_importance() — src/memory.py**
```python
def score_memory_importance(text: str, metadata: dict[str, Any] | None = None) -> float:
    normalized = normalize_text(text).lower()
    tokens = content_tokens(normalized)
    metadata = metadata or {}

    score = 0.2
    if len(tokens) >= 8:
        score += 0.2
    if any(marker in normalized for marker in ("remember", "prefers", "prefer", "always", "never", "important")):
        score += 0.25
    if any(character.isdigit() for character in normalized):
        score += 0.1
    if metadata.get("category") in {"preference", "constraint", "fact"}:
        score += 0.15
    if metadata.get("source") == "user":
        score += 0.1
    return round(min(score, 1.0), 3)
```

**코드 읽기 포인트**
- 초기 점수 0.2에서 시작해 정보성 힌트가 많을수록 가산점을 준다.
- `remember`, `prefers`, `important` 같은 표지가 있으면 장기 가치가 높다고 본다.
- 숫자와 user-origin 정보도 중요도 가산 요소다.
- 즉 이 구현은 완벽한 학습 모델이 아니라 "inspectable heuristic"이라는 점이 교육적으로 중요하다.

**결과 해석 가이드**
- importance score가 threshold 근처에 몰리면 정책이 너무 애매할 수 있다.
- 실무에서는 이 점수 함수를 telemetry와 함께 계속 조정하게 된다.


In [ ]:
candidates = [
    'Remember: Mina prefers concise rollout summaries with dates first.',
    'The user said thanks.',
    'Important: Team Apollo is piloting the scheduling template in Seoul.',
]
importance_frame = pd.DataFrame(
    {
        'candidate': candidates,
        'importance_score': [score_memory_importance(text, {'category': 'fact', 'source': 'user'}) for text in candidates],
    }
)
importance_frame


## 메모리 업데이트 전략(Memory Update Strategy)

이제 실제 저장 결정을 내려 본다. `selective_memory_update()`는 importance score를 계산하고, threshold를 넘을 때만 long-term과 vector memory에 기록한다. 즉 "저장 정책"을 코드로 드러내는 함수다.

**목적**
- importance score를 실제 저장 의사결정으로 연결한다.

**핵심 로직**
- 점수가 threshold 미만이면 즉시 `stored=False`로 반환한다.
- threshold 이상이면 long-term과 vector memory에 각각 저장하고 `stored_in`에 기록한다.

**주요 파라미터/변수**
- `text`: 저장 후보 문장
- `key`: 장기 메모리 key
- `threshold=0.55`: 현재 저장 기준선

**실제 소스 코드: selective_memory_update() — src/memory.py**
```python
def selective_memory_update(
    text: str,
    key: str,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    threshold: float = 0.55,
    category: str = "fact",
    metadata: dict[str, Any] | None = None,
) -> dict[str, Any]:
    metadata = metadata or {}
    effective_metadata = {"category": category, **metadata}
    importance_score = score_memory_importance(text, effective_metadata)
    decision = {
        "key": key,
        "text": normalize_text(text),
        "category": category,
        "importance_score": importance_score,
        "stored": importance_score >= threshold,
        "stored_in": [],
    }
    if importance_score < threshold:
        return decision

    if long_term_memory is not None:
        long_term_memory.store_memory(
            key=key,
            value=text,
            category=category,
            metadata=metadata,
            importance=importance_score,
        )
        decision["stored_in"].append("long_term")

    if vector_memory is not None:
        vector_memory.add_memory(
            text=text,
            metadata={"key": key, "category": category, **metadata},
            importance=importance_score,
        )
        decision["stored_in"].append("vector")

    return decision
```

**코드 읽기 포인트**
- `effective_metadata = {"category": category, **metadata}`: 점수 함수가 category를 항상 보게 만든다.
- `stored_in`을 결과에 남겨 어떤 메모리 채널에 저장됐는지 즉시 확인할 수 있다.
- 이 결정 결과를 그대로 표로 보여 주기 때문에, memory policy가 black box가 아니다.

**결과 해석 가이드**
- `stored=False`가 나왔다고 실패가 아니다. 오히려 low-signal memory를 차단했다는 뜻일 수 있다.
- 중요도 높은 메모리만 남아야 later retrieval이 깨끗해진다.

**💡 면접 포인트**
- "Memory는 recall보다 precision이 더 중요할 때가 많다. 쓸모없는 기억을 안 남기는 것이 품질에 직결된다"고 설명할 수 있다.


In [ ]:
update_decisions = [
    selective_memory_update(
        text='Remember: Mina prefers concise rollout summaries with dates first.',
        key='mina_style_rule',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='preference',
        metadata={'source': 'user'},
    ),
    selective_memory_update(
        text='The user said thanks.',
        key='low_signal_event',
        long_term_memory=reloaded_long_term,
        vector_memory=vector_memory,
        threshold=0.55,
        category='event',
        metadata={'source': 'user'},
    ),
]
pd.DataFrame(update_decisions)


## Agent workflow 안에서의 memory

메모리 클래스가 따로 존재하는 것만으로는 agent memory가 아니다. 실제 질문 처리 흐름 안에서 `retrieve_memories`와 `update_memory`가 언제 끼어드는지가 중요하다. 아래 셀은 memory-aware workflow를 end-to-end로 돌려, 최종 답변뿐 아니라 memory retrieval/update 흔적도 함께 본다.

**목적**
- 문서 검색과 메모리 검색, 답변 후 메모리 업데이트가 한 workflow에서 어떻게 연결되는지 본다.

**핵심 로직**
- `run_workflow_with_memory()`는 기본 workflow에 `retrieve_memories`와 `update_memory` node를 추가한다.
- `include_memories_in_context=False`면 메모리는 trace에는 남지만 문서 context에는 직접 합치지 않는다.
- `update_memory=True`면 실행 후 user query와 final answer가 short-term memory 등에 반영될 수 있다.

**주요 파라미터/변수**
- `memory_top_k`: 가져올 메모리 수
- `include_memories_in_context`: retrieved docs에 메모리를 pseudo-doc로 섞을지 여부
- `memory_threshold`: 저장 threshold

**실제 소스 코드: run_workflow_with_memory() — src/workflow.py**
```python
def run_workflow_with_memory(
    query: str,
    retriever: Any,
    short_term_memory: ShortTermMemory | None = None,
    long_term_memory: LongTermMemory | None = None,
    vector_memory: VectorMemory | None = None,
    top_k: int = DEFAULT_TOP_K,
    memory_top_k: int = 3,
    include_memories_in_context: bool = False,
    update_memory: bool = True,
    memory_threshold: float = 0.55,
    trace_path: Path | None = None,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> AgentState:
    state = create_initial_state(query)
    effective_llm_client = llm_client or (OllamaClient() if use_llm else None)

    try:
        _execute_workflow_steps(
            state,
            MEMORY_WORKFLOW_STEPS,
            retriever=retriever,
            top_k=top_k,
            use_llm=use_llm,
            llm_client=effective_llm_client,
            short_term_memory=short_term_memory,
            long_term_memory=long_term_memory,
            vector_memory=vector_memory,
            memory_top_k=memory_top_k,
            include_memories_in_context=include_memories_in_context,
            update_memory=update_memory,
            memory_threshold=memory_threshold,
        )
    except Exception as error:  # pragma: no cover - defensive path
        record_error(state, str(error))
        error_start = time.perf_counter()
        append_trace(
            state,
            "workflow_error",
            {
                "inputs": {"last_completed_node": state["trace"][-1]["node"] if state["trace"] else None},
                "outputs": {"error": str(error)},
                "latency": round(time.perf_counter() - error_start, 6),
            },
        )
        state["final_answer"] = "Workflow execution failed."
        state["final_status"] = "failed"

    if trace_path is not None:
        write_json(trace_path, state)

    return state
```

**코드 읽기 포인트**
- 기본 `WORKFLOW_STEPS`가 아니라 `MEMORY_WORKFLOW_STEPS`를 사용한다.
- `_execute_workflow_steps()`가 step 종류에 따라 retriever, llm_client, memory object를 필요한 node에만 주입한다.
- 즉 memory integration도 기존 workflow contract를 깨지 않고 additive하게 붙어 있다.

**결과 해석 가이드**
- `retrieved_memories`와 `memory_updates`가 0보다 크면 메모리 경로가 실제로 실행된 것이다.
- 최종 답변이 같더라도 trace 안에 memory node가 보이면 memory-aware run과 baseline run을 구분할 수 있다.


In [ ]:
memory_state = run_workflow_with_memory(
    'When does the organization-wide rollout begin?',
    retriever=retriever,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    include_memories_in_context=False,
    update_memory=True,
)
{
    'final_status': memory_state['final_status'],
    'final_answer': memory_state['final_answer'],
    'retrieved_memories': len(memory_state['retrieved_memories']),
    'memory_updates': len(memory_state['memory_updates']),
}


## 시각화(Visualization)

메모리를 디버깅하려면 저장소 상태를 한 번에 보는 화면이 필요하다. `display_memory()`는 short-term, long-term, vector memory를 각각 표로 보여 줘서, 어떤 정보가 어느 채널에 들어가 있는지 즉시 확인하게 해 준다.

**목적**
- 세 종류 메모리를 한 번에 inspection한다.

**핵심 로직**
- 각 memory object의 `to_frame()` 결과를 묶어 notebook에서 display한다.

**결과 해석 가이드**
- 세 표의 역할이 다르므로 같은 정보가 여러 채널에 중복될 수 있다. 중요한 것은 의도한 채널에 들어갔는지다.


In [ ]:
display_memory(
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
)


## 시각화(Visualization)

trace 관점에서 memory를 보는 것도 중요하다. 메모리 동작이 "어딘가에서 몰래 일어나는 부가 기능"이 아니라, workflow 안의 명시적 node라는 사실이 trace에 드러나야 시스템을 신뢰할 수 있다.

**목적**
- memory node가 workflow trace에 어떻게 나타나는지 읽는다.

**결과 해석 가이드**
- trace에 `retrieve_memories`, `update_memory`가 보이면 memory integration이 성공적으로 연결된 것이다.
- latency까지 함께 보면 메모리 기능이 성능에 주는 비용도 추정할 수 있다.


In [ ]:
display_trace(memory_state['trace'])


## 실험

이제 같은 질문에 대해 문서만 쓴 경우와 문서+메모리를 함께 쓴 경우를 비교한다. 이 실험의 핵심은 답변 문장 자체보다, context bundle 안에 어떤 정보가 추가되었는지 보는 것이다. memory는 정답 생성보다 context shaping에서 먼저 효과를 드러낸다.

**목적**
- memory augmentation이 context 구성을 어떻게 바꾸는지 비교한다.

**결과 해석 가이드**
- `memory_count`가 늘고 `combined_context`에 선호/규칙이 들어오면, 메모리가 답변 스타일과 초점에 개입할 준비가 된 것이다.


In [ ]:
query = 'How should I answer rollout timing for Mina?'
no_memory_bundle = build_memory_augmented_context(query=query, retrieved_docs=retrieved_docs, top_k=3)
with_memory_bundle = build_memory_augmented_context(
    query=query,
    retrieved_docs=retrieved_docs,
    short_term_memory=short_term,
    long_term_memory=reloaded_long_term,
    vector_memory=vector_memory,
    top_k=3,
)
pd.DataFrame(
    [
        {
            'mode': 'docs_only',
            'memory_count': len(no_memory_bundle['retrieved_memories']),
            'combined_context': str(no_memory_bundle['combined_context']),
        },
        {
            'mode': 'docs_plus_memory',
            'memory_count': len(with_memory_bundle['retrieved_memories']),
            'combined_context': str(with_memory_bundle['combined_context']),
        },
    ]
)


## 실험 2: memory pollution

memory pollution은 관련 없는 기억이 retrieval 상위권에 올라오는 현상이다. 메모리를 많이 저장하는 것이 왜 위험한지 보여주는 가장 좋은 예다. 사용자의 진짜 요구는 rollout timing인데, snack 주문이나 poster 색상 같은 잡음이 같이 떠오르면 answer relevance가 떨어진다.

**목적**
- 불필요한 기억이 retrieval quality를 어떻게 해치는지 본다.

**결과 해석 가이드**
- 상위 검색 결과에 noise memory가 섞이면 selective update나 retrieval ranking을 강화해야 한다.
- vector memory는 semantic retrieval이지만, semantic similarity가 곧 task relevance를 보장하지는 않는다.


In [ ]:
polluted_memory = VectorMemory()
polluted_memory.add_memory('Mina prefers concise rollout answers that begin with the exact launch date.', metadata={'kind': 'preference'}, importance=0.9)
polluted_memory.add_memory('Mina is ordering snacks for the rollout celebration.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('Rollout posters should use the coral brand palette.', metadata={'kind': 'noise'}, importance=0.4)
polluted_memory.add_memory('The rollout FAQ should mention the May 5, 2025 launch date.', metadata={'kind': 'fact'}, importance=0.8)
pd.DataFrame(polluted_memory.search('How should I answer rollout timing for Mina?', top_k=4))


## 실험 3: memory summarization

최근 대화를 전부 넣는 대신 요약(summary)으로 압축하는 것도 메모리 전략이다. short-term memory가 길어질수록 raw event를 모두 다시 주입하는 것보다, 중요한 점만 요약해 long-term memory에 남기는 편이 효율적일 때가 많다.

**목적**
- 여러 event를 짧은 summary로 압축해 저장하는 패턴을 본다.

**핵심 로직**
- `summarize_memory_events()`가 최근 event를 문자열 하나로 압축한다.
- 이 summary를 다시 long-term memory에 저장해 future retrieval 후보로 만든다.

**결과 해석 가이드**
- 요약 텍스트가 너무 추상적이면 retrieval에서 힘을 못 쓰고, 너무 길면 압축 이점이 줄어든다.


In [ ]:
summary_text = summarize_memory_events(short_term.events, limit=5)
reloaded_long_term.store_memory('recent_memory_summary', summary_text, category='summary')
pd.DataFrame(
    {
        'summary_text': [summary_text],
        'stored_summary': [reloaded_long_term.retrieve_memory('recent_memory_summary')['value']],
    }
)


## 결과 해석

마지막 분석 표는 세 종류 메모리와 workflow integration을 한눈에 요약한다. 여기서 봐야 할 것은 메모리 자체의 존재가 아니라, memory retrieval과 update가 실제 실행 흔적으로 남았는지, 그리고 어떤 channel이 가장 유용한지다.

**목적**
- 메모리 시스템의 효과를 채널별로 읽는다.

**결과 해석 가이드**
- short-term observation은 최근성, long-term observation은 persistence, vector memory observation은 semantic recall 품질로 읽으면 된다.
- `trace contains memory nodes=True`는 메모리 기능이 실제 workflow에 연결됐다는 가장 직접적인 신호다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'theme': 'short_term_memory',
            'observation': f"buffer size after workflow run: {len(short_term.events)} events",
        },
        {
            'theme': 'long_term_memory',
            'observation': f"persisted items: {len(reloaded_long_term.list_items())}",
        },
        {
            'theme': 'vector_memory',
            'observation': f"top memory hit: {vector_search_results[0]['text'] if vector_search_results else 'none'}",
        },
        {
            'theme': 'workflow_integration',
            'observation': f"trace contains memory nodes: {any(entry['node'] == 'retrieve_memories' for entry in memory_state['trace'])}",
        },
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 agent memory는 하나의 저장소가 아니라, 역할이 다른 여러 메모리 채널의 조합이라는 점을 확인했다. short-term memory는 최근 대화 버퍼, long-term memory는 지속 저장되는 선호와 사실, vector memory는 의미 유사 검색용 인덱스다. 그리고 이들을 실제 agent workflow 안에서 retrieval과 update node로 연결해야 비로소 "메모리를 쓰는 agent"가 된다.

동시에 모든 것을 기억하면 안 된다는 점도 분명해졌다. selective memory update가 없으면 memory pollution이 생기고, retrieval noise가 늘며, 오래된 정보가 새로운 답변을 오염시킬 수 있다. 결국 memory 설계의 핵심은 저장량이 아니라 선택과 정리다.

**💡 면접 포인트**
- "Short-term / long-term / vector memory를 분리하면 각 채널의 역할과 failure mode를 설명하기 쉬워진다."
- "모든 대화를 다 기억하는 것은 품질 향상이 아니라 retrieval noise 증가로 이어질 수 있다."
- "Memory는 workflow 안에서 retrieve/update node로 명시적으로 드러나야 디버깅과 평가가 가능하다."
